In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from src.modeling import ModelingData
from sklearn.model_selection import train_test_split

 ### Load cleaned data

In [2]:
df = pd.read_csv('../data/insurance_data_cleaned.csv')
df['TransactionMonth'] = pd.to_datetime(df['TransactionMonth'])

# Create required columns
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

print(f"Loaded {len(df):,} rows")
print(f"Columns: {df.shape[1]}")
print(f"HasClaim: {df['HasClaim'].sum():,} claims ({df['HasClaim'].mean():.4%})")

Loaded 988,797 rows
Columns: 48
HasClaim: 2,613 claims (0.2643%)


IMPUTE GENDER FROM TITLE

In [3]:
df = ModelingData.impute_gender_from_title(df)
print(df['Gender'].value_counts())

Gender
Male             922262
Female            65725
Not specified       810
Name: count, dtype: int64


Filter to policies with claims

In [4]:

severity_df = df[df['TotalClaims'] > 0].copy()
print(f"Original dataset: {len(df):,} rows")
print(f"Severity dataset: {len(severity_df):,} rows ({len(severity_df)/len(df):.2%} of data)")
print(f"Claims range: R{severity_df['TotalClaims'].min():.2f} to R{severity_df['TotalClaims'].max():.2f}")

Original dataset: 988,797 rows
Severity dataset: 2,613 rows (0.26% of data)
Claims range: R139.04 to R393092.11


### Feature Engineering

In [5]:
severity_df = ModelingData.engineer_features(severity_df)

# Check new features
print("New features created:")
print(f"vehicle_age - min: {severity_df['vehicle_age'].min()}, max: {severity_df['vehicle_age'].max()}")
print(f"policy_duration - min: {severity_df['policy_duration'].min()}, max: {severity_df['policy_duration'].max()}")
print("\nSample data:")
display(severity_df[['RegistrationYear', 'vehicle_age', 'TransactionMonth', 'policy_duration']].head())

New features created:
vehicle_age - min: 0, max: 19
policy_duration - min: 0, max: 638

Sample data:


,RegistrationYear,vehicle_age,TransactionMonth,policy_duration
281,2006,9,2015-03-01,153
1557,2014,1,2015-04-01,122
1776,2009,6,2014-10-01,304
1940,2014,1,2015-04-01,122
2069,2008,7,2015-03-01,153


In [6]:
#  Ensure no missing values
severity_df = ModelingData.handle_missing_values(severity_df)
print(f"Missing values after handling: {severity_df.isnull().sum().sum()}")

Missing values after handling: 0


### Separate Features and Target

In [7]:
# Get X (features) and y (target)
X, y, feature_cols = ModelingData.get_feature_columns(severity_df, target_col='TotalClaims')

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFirst few features: {feature_cols[:5]}")

Features shape: (2613, 40)
Target shape: (2613,)

First few features: ['IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language']


### Identify Column Types

In [8]:
# Identify numerical and categorical columns
numerical_cols, categorical_cols = ModelingData.get_column_types(X)

print(f"Numerical columns: {len(numerical_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"\nNumerical: {numerical_cols[:5]}")
print(f"Categorical: {categorical_cols[:5]}")

Numerical columns: 10
Categorical columns: 29

Numerical: ['PostalCode', 'mmcode', 'Cylinders', 'cubiccapacity', 'kilowatts']
Categorical: ['Citizenship', 'LegalType', 'Title', 'Language', 'Bank']


### Train/Test Split

In [9]:
# Split data 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train):,} policies")
print(f"Test set: {len(X_test):,} policies")
print(f"Train mean claim: R{y_train.mean():.2f}")
print(f"Test mean claim: R{y_test.mean():.2f}")

Training set: 2,090 policies
Test set: 523 policies
Train mean claim: R23745.31
Test mean claim: R21793.82


###  PREPARE DATA FOR EACH MODEL TYPE

In [10]:

# For Random Forest & XGBoost (Label Encoding)
X_train_label, X_test_label, encoders, scaler = ModelingData.prepare_label_encoding_data(
    X_train, X_test, numerical_cols, categorical_cols
)

# For Linear Regression (One-Hot Encoding)
X_train_lr, X_test_lr, onehot, scaler_lr = ModelingData.prepare_onehot_data(
    X_train, X_test, numerical_cols, categorical_cols
)

print(f"Label encoding features: {X_train_label.shape[1]}")
print(f"One-hot encoding features: {X_train_lr.shape[1]}")

Label encoding features: 40
One-hot encoding features: 320
